# 07 - Primeiro Turno vs Segundo Turno

Times que arrancam no 1º turno mantêm no 2º? Existe correlação forte?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)
df = df[df["ano"] >= 2003].copy()

In [2]:
def pontos_por_turno(df_season):
    """Calcula pontos no 1º e 2º turno para cada time."""
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    pts_t1 = {t: 0 for t in teams}
    pts_t2 = {t: 0 for t in teams}
    
    for _, jogo in df_season.iterrows():
        m, v = jogo["mandante"], jogo["visitante"]
        gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
        if pd.isna(gm) or pd.isna(gv):
            continue
        gm, gv = int(gm), int(gv)
        turno = pts_t1 if jogo["rodata"] <= 19 else pts_t2
        
        if gm > gv:
            turno[m] += 3
        elif gm < gv:
            turno[v] += 3
        else:
            turno[m] += 1
            turno[v] += 1
    
    return [{"time": t, "pts_turno1": pts_t1[t], "pts_turno2": pts_t2[t]} for t in teams]

# Calcular para todas as temporadas
todos = []
for ano in sorted(df["ano"].unique()):
    df_s = df[df["ano"] == ano]
    if df_s["rodata"].max() < 38:
        continue
    turnos = pontos_por_turno(df_s)
    for t in turnos:
        t["ano"] = ano
        # Verificar se foi campeão (maior pontuação total)
        t["pts_total"] = t["pts_turno1"] + t["pts_turno2"]
    todos.extend(turnos)

df_turnos = pd.DataFrame(todos)
# Marcar campeão de cada ano
for ano in df_turnos["ano"].unique():
    mask = df_turnos["ano"] == ano
    max_pts = df_turnos.loc[mask, "pts_total"].max()
    df_turnos.loc[mask, "campeao"] = df_turnos.loc[mask, "pts_total"] == max_pts

In [3]:
# Scatter plot: 1º turno x 2º turno
corr = df_turnos["pts_turno1"].corr(df_turnos["pts_turno2"])

fig = px.scatter(
    df_turnos, x="pts_turno1", y="pts_turno2",
    color=df_turnos["campeao"].map({True: "Campeão", False: "Demais"}),
    color_discrete_map={"Campeão": "#e74c3c", "Demais": "#3498db"},
    opacity=0.6,
    hover_data=["time", "ano"],
    title=f"Desempenho no 1º Turno vs 2º Turno<br><sup>Correlação: r = {corr:.3f} | Brasileirão (2003-presente)</sup>",
    labels={"pts_turno1": "Pontos no 1º Turno", "pts_turno2": "Pontos no 2º Turno",
            "color": ""}
)

# Linha de tendência
z = np.polyfit(df_turnos["pts_turno1"], df_turnos["pts_turno2"], 1)
x_line = np.linspace(df_turnos["pts_turno1"].min(), df_turnos["pts_turno1"].max(), 100)
fig.add_trace(go.Scatter(x=x_line, y=np.polyval(z, x_line),
                         mode="lines", line=dict(color="gray", dash="dash"),
                         name="Tendência", showlegend=False))

# Linha diagonal (desempenho igual nos dois turnos)
fig.add_trace(go.Scatter(x=[0, 57], y=[0, 57],
                         mode="lines", line=dict(color="lightgray", dash="dot"),
                         name="Igual nos 2 turnos", showlegend=False))

fig.update_layout(template="plotly_white", width=800, height=600)

fig.show()
_path = "../charts/turnos.html"
fig.write_html(_path, include_plotlyjs="cdn")

# Adiciona descrição estatística
_desc = "Scatter plot da correlação entre pontos no 1º e 2º turno (r ≈ 0.384), indicando correlação positiva moderada. A maioria dos pontos situa-se abaixo da diagonal, sugerindo tendência geral de queda de rendimento no returno."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/turnos.html")

Gráfico exportado: charts/turnos.html


In [4]:
# Quais times melhoram mais no 2º turno? (média)
GRANDES = ["Flamengo", "Palmeiras", "Corinthians", "São Paulo", "Santos",
           "Grêmio", "Internacional", "Cruzeiro", "Atlético-MG",
           "Fluminense", "Vasco", "Botafogo"]

grandes = df_turnos[df_turnos["time"].isin(GRANDES)].copy()
grandes["diff"] = grandes["pts_turno2"] - grandes["pts_turno1"]

media_diff = grandes.groupby("time")["diff"].mean().sort_values().reset_index()

fig2 = px.bar(media_diff, x="diff", y="time", orientation="h",
              title="Diferença Média de Pontos: 2º Turno - 1º Turno<br><sup>Positivo = melhora no returno | Negativo = piora</sup>",
              labels={"diff": "Diferença de Pontos (2ºT - 1ºT)", "time": ""},
              color="diff",
              color_continuous_scale="RdYlGn",
              color_continuous_midpoint=0)
fig2.update_layout(template="plotly_white", width=800, height=500, showlegend=False)

fig2.show()
_path = "../charts/turnos_diff.html"
fig2.write_html(_path, include_plotlyjs="cdn")

# Adiciona descrição estatística
_desc = "Diferença média de pontos entre 2º e 1º turno por clube, onde valores positivos indicam tendência de melhoria no returno."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/turnos_diff.html")

Gráfico exportado: charts/turnos_diff.html
